## **01. Geração de Dados Sintéticos — Evasão Escolar**

Este notebook gera o dataset sintético baseado nas estatísticas reais do Censo Escolar (INEP) e SAEB.

**Por que dados sintéticos?**  
Os microdados do INEP e SAEB são arquivos de vários GBs, o que inviabiliza a reprodutibilidade imediata do projeto. O dataset gerado aqui preserva as distribuições e relações estatísticas reais documentadas nos relatórios públicos do INEP, como:
- Taxa de evasão nacional ~11-22% no Ensino Médio
- Correlação entre distorção idade-série e abandono
- Diferença de evasão por turno (noturno > matutino)

**Fonte de referência:** [INEP — Censo Escolar](https://www.gov.br/inep/pt-br/acesso-a-informacao/dados-abertos/microdados/censo-escolar)

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Seed global para reprodutibilidade
RANDOM_STATE = 42
N_AMOSTRAS   = 15_000
DATA_DIR     = Path('../data')
DATA_DIR.mkdir(exist_ok=True)

## **Função Geradora:**

In [2]:
def gerar_dataset_evasao(n: int = 15_000, seed: int = 42) -> pd.DataFrame:
    """
    Gera dataset sintético de evasão escolar no Ensino Médio público brasileiro.

    As distribuições são baseadas nos relatórios:
      - INEP: Censo Escolar 2022 — Taxas de Rendimento
      - SAEB 2021 — Proficiências médias
      - IBGE: PNAD Contínua 2022 — Renda familiar

    Parâmetros
    ----------
    n : int
        Número de alunos a simular.
    seed : int
        Semente para reprodutibilidade.

    Retorna
    -------
    pd.DataFrame com colunas demográficas, escolares, acadêmicas,
    socioeconômicas e a variável-alvo `evasao`.
    """
    rng = np.random.default_rng(seed)

    # ── Variáveis demográficas ─────────────────────────────────────────────
    serie = rng.choice(['1ano', '2ano', '3ano'], size=n,
                       p=[0.37, 0.34, 0.29])          # proporção real INEP 2022
    turno = rng.choice(['matutino', 'vespertino', 'noturno'], size=n,
                       p=[0.45, 0.30, 0.25])

    # Idade esperada por série; distorção clipeada em [0, 5]
    idade_esperada = np.where(serie == '1ano', 15,
                     np.where(serie == '2ano', 16, 17))
    distorcao = np.clip(rng.normal(0.8, 1.2, n).astype(int), 0, 5)
    idade     = np.clip(idade_esperada + distorcao, 14, 24)

    sexo     = rng.choice(['M', 'F'], size=n, p=[0.50, 0.50])
    raca_cor = rng.choice(['Branca', 'Parda', 'Preta', 'Amarela', 'Indígena'],
                          size=n, p=[0.43, 0.46, 0.09, 0.01, 0.01])

    # ── Variáveis socioeconômicas ──────────────────────────────────────────
    renda_familiar  = np.clip(rng.lognormal(7.1, 0.8, n), 500, 20_000).astype(int)
    escol_pai       = rng.choice([0, 1, 2, 3], n, p=[0.35, 0.30, 0.25, 0.10])
    escol_mae       = rng.choice([0, 1, 2, 3], n, p=[0.30, 0.30, 0.28, 0.12])
    trabalha        = rng.choice([0, 1], n, p=[0.65, 0.35])
    # Alunos do noturno têm maior probabilidade de trabalhar
    trabalha        = np.where(turno == 'noturno',
                               rng.choice([0, 1], n, p=[0.35, 0.65]), trabalha)

    # Variáveis acadêmicas
    # Médias baseadas no SAEB: ~260 pts em Mat, ~270 em Port (escala [0,10])
    nota_port = np.clip(rng.normal(6.0, 2.0, n), 0, 10)
    nota_mat  = np.clip(rng.normal(5.8, 2.1, n), 0, 10)

    # Faltas correlacionadas com distorção
    faltas = np.clip(
        rng.poisson(10, n) + (distorcao * 3) + (trabalha * 5),
        0, 60
    ).astype(int)

    repeticoes = np.clip(distorcao // 2, 0, 3).astype(int)

    # Variável-alvo: evasão
    # Probabilidade baseada em fatores de risco documentados
    prob_evasao = (
        0.25 * (nota_mat  < 5).astype(float) +
        0.20 * (nota_port < 5).astype(float) +
        0.30 * (distorcao > 1).astype(float) +
        0.25 * (faltas    > 20).astype(float) +
        0.10 * (trabalha == 1).astype(float) +
        0.05 * (turno == 'noturno').astype(float)
    )
    prob_evasao = np.clip(
        prob_evasao + rng.normal(0, 0.08, n), 0, 1
    )
    evasao = (prob_evasao > 0.40).astype(int)

    # Montar DataFrame
    df = pd.DataFrame({
        'idade'                : idade,
        'sexo'                 : sexo,
        'raca_cor'             : raca_cor,
        'serie'                : serie,
        'turno'                : turno,
        'nota_portugues'       : nota_port.round(2),
        'nota_matematica'      : nota_mat.round(2),
        'distorcao_idade_serie': distorcao,
        'faltas_anuais'        : faltas,
        'repeticoes_anteriores': repeticoes,
        'trabalha'             : trabalha,
        'renda_familiar'       : renda_familiar,
        'escolaridade_pai'     : escol_pai,
        'escolaridade_mae'     : escol_mae,
        'evasao'               : evasao,
    })

    return df

## **Gerar e Salvar Dataset:**

In [3]:
df = gerar_dataset_evasao(n=N_AMOSTRAS, seed=RANDOM_STATE)

# Salvar
df.to_csv(DATA_DIR / 'evasao_escolar.csv', index=False)
print(f'Dataset salvo em: {DATA_DIR / "evasao_escolar.csv"}')
print(f'Shape: {df.shape}')
print(f'Taxa de evasão: {df["evasao"].mean():.1%}')

Dataset salvo em: ..\data\evasao_escolar.csv
Shape: (15000, 15)
Taxa de evasão: 26.5%


In [4]:
print(df.dtypes)
print('\n', df.head())

idade                      int64
sexo                      object
raca_cor                  object
serie                     object
turno                     object
nota_portugues           float64
nota_matematica          float64
distorcao_idade_serie      int64
faltas_anuais              int64
repeticoes_anteriores      int64
trabalha                   int64
renda_familiar             int64
escolaridade_pai           int64
escolaridade_mae           int64
evasao                     int64
dtype: object

    idade sexo raca_cor serie       turno  nota_portugues  nota_matematica  \
0     17    F    Preta  3ano    matutino            7.06             6.36   
1     16    M    Parda  2ano    matutino            1.63             4.07   
2     17    M   Branca  3ano    matutino            5.46             5.37   
3     16    M   Branca  2ano     noturno            9.13             6.69   
4     16    M   Branca  1ano  vespertino            4.46             8.83   

   distorcao_idade_serie  